# TakeOff.ai — train the spaces model on Kaggle

This notebook downloads the private, validated ResPlan-derived `spaces-v1` archive and runs a one-epoch YOLO segmentation smoke test.

Before running it, enable **Internet** and select **GPU** under Kaggle's **Settings → Session options**. Keep the notebook private because it accesses a private dataset.


## 1. Confirm the GPU


In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi', '-L'], check=True)
assert torch.cuda.is_available(), 'No CUDA GPU detected. Enable a GPU in Kaggle Session options.'
print('PyTorch:', torch.__version__, '| CUDA device:', torch.cuda.get_device_name(0))


## 2. Download the TakeOff code
The GitHub repository is public, so no GitHub token is required.


In [ ]:
import os
import subprocess
from pathlib import Path

repo_dir = Path('/kaggle/working/TakeOff')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Siddartha-DevOps/TakeOff.git', str(repo_dir)], check=True)
backend_dir = repo_dir / 'app' / 'backend'
os.chdir(backend_dir)
print('Working directory:', Path.cwd())


## 3. Install and check the ML dependencies


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub'], check=True)
subprocess.run([sys.executable, '-m', 'ml.preflight'], check=False)


## 4. Download and strictly validate the private ResPlan dataset
In Kaggle, open **Add-ons → Secrets**, create `HF_TOKEN`, paste a Hugging Face token with read access, and attach it to this notebook. Never paste the token into a code cell.


In [ ]:
import hashlib
import os
import subprocess
import sys
import tarfile
from pathlib import Path
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
assert hf_token, 'Attach the HF_TOKEN secret to this Kaggle notebook.'
archive = hf_hub_download(
    repo_id='Siddartha96/takeoff-spaces-v1',
    filename='spaces-v1-097a39c19c57a208.tar.gz',
    repo_type='dataset',
    token=hf_token,
)
expected_sha256 = '2838390abf556d7f6bb23d2918363c82875e7b4f3185a9852c55c4b9fb708c02'
actual_sha256 = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
assert actual_sha256 == expected_sha256, f'Archive checksum mismatch: {actual_sha256}'

target = Path('data').resolve()
target.mkdir(exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    for member in bundle.getmembers():
        destination = (target / member.name).resolve()
        assert destination == target or str(destination).startswith(str(target) + os.sep), member.name
    bundle.extractall(target)

print('Verified archive extracted:', archive)
subprocess.run([sys.executable, '-m', 'ml.datasets.validate_spaces', '--data',
                'data/spaces_v1/data.yaml', '--require-groups'], check=True)
subprocess.run([sys.executable, '-m', 'ml.preflight', '--data',
                'data/spaces_v1/data.yaml', '--require', 'train'], check=True)


## 5. Run the one-epoch smoke test
This verifies the complete GPU training pipeline; it is not an accuracy result.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'ml.training.run_training',
                '--data', 'data/spaces_v1/data.yaml', '--task', 'spaces',
                '--smoke', '--no-promote'], check=True)


## 6. Kaggle-budget full training — keep disabled until the smoke logs are reviewed
The ResPlan renders are 768×768. This first full run uses 640px, batch 8, and 12 epochs so it fits one free T4 session without wasteful upscaling.


In [ ]:
import subprocess
import sys

RUN_FULL_TRAINING = False
EPOCHS, IMAGE_SIZE, BATCH_SIZE = 12, 640, 8
if RUN_FULL_TRAINING:
    subprocess.run([sys.executable, '-m', 'ml.training.run_training',
                    '--data', 'data/spaces_v1/data.yaml', '--task', 'spaces',
                    '--epochs', str(EPOCHS), '--imgsz', str(IMAGE_SIZE),
                    '--batch', str(BATCH_SIZE)], check=True)
else:
    print('Stopped after smoke training. Share the smoke logs before enabling full training.')


## 7. Save trained weights as a Kaggle output
After an approved full run, this copies `best.pt` into `/kaggle/working/output`, which Kaggle preserves when you save a notebook version.


In [ ]:
import shutil
from pathlib import Path

weights = Path('models/best.pt')
output = Path('/kaggle/working/output/best.pt')
if weights.is_file():
    output.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, output)
    print('Saved Kaggle output:', output)
else:
    print('No promoted best.pt yet. Complete an approved full training run first.')
